In [1]:
import os
import math
import copy
import h5py
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# =========================
# paths / device
# =========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

load_path = "/home/sz4544/EEG-motor-imagery-main/project/train1800_raw_EEG.h5"
model_path = "/home/sz4544/EEG-motor-imagery-main/project/models/diffusion_train1800_v3_ema.pt"
scaler_path = "/home/sz4544/EEG-motor-imagery-main/project/models/diffusion_train1800_v3_scaler.npz"

os.makedirs("/home/sz4544/EEG-motor-imagery-main/project/models", exist_ok=True)

# =========================
# load data
# =========================
f = h5py.File(load_path, "r")
x_train = f["data"][:]
y_train = f["tasks"][:]
f.close()

print("raw x_train:", x_train.shape)
print("raw y_train:", y_train.shape)
print("raw labels:", np.unique(y_train, return_counts=True))

y_train = y_train.astype(np.int64)
if np.array_equal(np.unique(y_train), np.array([1, 2, 3, 4])):
    y_train = y_train - 1

print("remapped labels:", np.unique(y_train, return_counts=True))

x_train = x_train.astype(np.float32)

# per-channel normalization over train only
mean = x_train.mean(axis=(0, 1), keepdims=True)
std = x_train.std(axis=(0, 1), keepdims=True) + 1e-6

x_train = (x_train - mean) / std

np.savez(scaler_path, mean=mean, std=std)

print("standardized global mean:", x_train.mean())
print("standardized global std:", x_train.std())

# convert to (N, C, T) = (N, 64, 640)
x_train = np.transpose(x_train, (0, 2, 1))
print("diffusion input shape:", x_train.shape)

# inspect real single-sample stats in diffusion domain
real_means = np.array([np.mean(x_train[i]) for i in range(len(x_train))])
real_stds = np.array([np.std(x_train[i]) for i in range(len(x_train))])
real_maxabs = np.array([np.max(np.abs(x_train[i])) for i in range(len(x_train))])

print("REAL diffusion-domain sample stats:")
print("mean percentiles:", np.percentile(real_means, [0, 1, 5, 25, 50, 75, 95, 99, 100]))
print("std percentiles:", np.percentile(real_stds, [0, 1, 5, 25, 50, 75, 95, 99, 100]))
print("maxabs percentiles:", np.percentile(real_maxabs, [0, 1, 5, 25, 50, 75, 95, 99, 100]))

# =========================
# dataset
# =========================
class EEGDiffusionDataset(Dataset):
    def __init__(self, x, y):
        self.x = torch.tensor(x, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]

dataset = EEGDiffusionDataset(x_train, y_train)
loader = DataLoader(dataset, batch_size=32, shuffle=True, drop_last=True)

print("dataset size:", len(dataset))
print("num batches:", len(loader))

# =========================
# model
# =========================
class SinusoidalPosEmb(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, t):
        half_dim = self.dim // 2
        emb = math.log(10000) / max(half_dim - 1, 1)
        emb = torch.exp(torch.arange(half_dim, device=t.device) * -emb)
        emb = t[:, None] * emb[None, :]
        emb = torch.cat((emb.sin(), emb.cos()), dim=1)
        return emb

class ResBlock1D(nn.Module):
    def __init__(self, in_ch, out_ch, cond_dim, groups=8):
        super().__init__()

        self.norm1 = nn.GroupNorm(num_groups=min(groups, in_ch), num_channels=in_ch)
        self.conv1 = nn.Conv1d(in_ch, out_ch, kernel_size=3, padding=1)

        self.norm2 = nn.GroupNorm(num_groups=min(groups, out_ch), num_channels=out_ch)
        self.conv2 = nn.Conv1d(out_ch, out_ch, kernel_size=3, padding=1)

        self.cond_mlp = nn.Sequential(
            nn.SiLU(),
            nn.Linear(cond_dim, out_ch * 2)
        )

        if in_ch != out_ch:
            self.res_conv = nn.Conv1d(in_ch, out_ch, kernel_size=1)
        else:
            self.res_conv = nn.Identity()

    def forward(self, x, cond):
        h = self.conv1(F.silu(self.norm1(x)))

        scale_shift = self.cond_mlp(cond).unsqueeze(-1)   # (B, 2*out_ch, 1)
        scale, shift = torch.chunk(scale_shift, 2, dim=1)

        h = self.norm2(h)
        h = h * (1.0 + scale) + shift
        h = self.conv2(F.silu(h))

        return h + self.res_conv(x)

class SimpleCondUNet1D(nn.Module):
    def __init__(self, n_channels=64, n_classes=4, base=64, time_dim=128, cond_dim=256):
        super().__init__()

        # stronger conditioning
        self.time_emb = SinusoidalPosEmb(time_dim)
        self.time_mlp = nn.Sequential(
            nn.Linear(time_dim, cond_dim),
            nn.SiLU(),
            nn.Linear(cond_dim, cond_dim)
        )

        self.class_emb = nn.Embedding(n_classes, cond_dim)

        self.in_proj = nn.Conv1d(n_channels, base, kernel_size=3, padding=1)

        self.block1 = ResBlock1D(base, base, cond_dim)
        self.block1b = ResBlock1D(base, base, cond_dim)

        self.down1 = nn.Conv1d(base, base * 2, kernel_size=4, stride=2, padding=1)
        self.block2 = ResBlock1D(base * 2, base * 2, cond_dim)
        self.block2b = ResBlock1D(base * 2, base * 2, cond_dim)

        self.down2 = nn.Conv1d(base * 2, base * 4, kernel_size=4, stride=2, padding=1)
        self.mid1 = ResBlock1D(base * 4, base * 4, cond_dim)
        self.mid2 = ResBlock1D(base * 4, base * 4, cond_dim)

        self.up1 = nn.ConvTranspose1d(base * 4, base * 2, kernel_size=4, stride=2, padding=1)
        self.block3 = ResBlock1D(base * 4, base * 2, cond_dim)
        self.block3b = ResBlock1D(base * 2, base * 2, cond_dim)

        self.up2 = nn.ConvTranspose1d(base * 2, base, kernel_size=4, stride=2, padding=1)
        self.block4 = ResBlock1D(base * 2, base, cond_dim)
        self.block4b = ResBlock1D(base, base, cond_dim)

        self.out_norm = nn.GroupNorm(num_groups=min(8, base), num_channels=base)
        self.out_proj = nn.Conv1d(base, n_channels, kernel_size=3, padding=1)

    def forward(self, x, t, y):
        t_emb = self.time_emb(t)
        t_emb = self.time_mlp(t_emb)
        c_emb = self.class_emb(y)
        cond = t_emb + c_emb

        x0 = self.in_proj(x)

        x1 = self.block1(x0, cond)
        x1 = self.block1b(x1, cond)

        x2 = self.down1(x1)
        x2 = self.block2(x2, cond)
        x2 = self.block2b(x2, cond)

        x3 = self.down2(x2)
        x3 = self.mid1(x3, cond)
        x3 = self.mid2(x3, cond)

        x = self.up1(x3)
        x = torch.cat([x, x2], dim=1)
        x = self.block3(x, cond)
        x = self.block3b(x, cond)

        x = self.up2(x)
        x = torch.cat([x, x1], dim=1)
        x = self.block4(x, cond)
        x = self.block4b(x, cond)

        x = self.out_proj(F.silu(self.out_norm(x)))
        return x

# =========================
# diffusion schedule
# =========================
T = 1000
betas = torch.linspace(1e-4, 0.02, T, device=device)
alphas = 1.0 - betas
alphas_cumprod = torch.cumprod(alphas, dim=0)

def q_sample(x0, t, noise):
    a = alphas_cumprod[t].view(-1, 1, 1)
    return torch.sqrt(a) * x0 + torch.sqrt(1.0 - a) * noise

# =========================
# EMA
# =========================
class EMA:
    def __init__(self, beta=0.999):
        self.beta = beta

    def update_model_average(self, ema_model, model):
        with torch.no_grad():
            for ema_param, param in zip(ema_model.parameters(), model.parameters()):
                ema_param.data = self.beta * ema_param.data + (1.0 - self.beta) * param.data

# =========================
# train
# =========================
model = SimpleCondUNet1D().to(device)
ema_model = copy.deepcopy(model).to(device)
ema_model.eval()

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
ema = EMA(beta=0.999)

epochs = 60
grad_clip = 1.0

model.train()

for epoch in range(epochs):
    losses = []

    for xb, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)

        t = torch.randint(0, T, (xb.size(0),), device=device)
        noise = torch.randn_like(xb)
        xt = q_sample(xb, t, noise)

        pred = model(xt, t.float(), yb)
        loss = F.mse_loss(pred, noise)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        optimizer.step()

        ema.update_model_average(ema_model, model)

        losses.append(loss.item())

    print(f"epoch {epoch+1}/{epochs} loss={np.mean(losses):.6f}")

# save EMA weights, not raw model weights
torch.save(ema_model.state_dict(), model_path)
print("saved EMA model:", model_path)
print("saved scaler:", scaler_path)

cuda
raw x_train: (1800, 640, 64)
raw y_train: (1800,)
raw labels: (array([1., 2., 3., 4.]), array([450, 450, 450, 450]))
remapped labels: (array([0, 1, 2, 3]), array([450, 450, 450, 450]))
standardized global mean: -2.8500283e-06
standardized global std: 0.9842581
diffusion input shape: (1800, 64, 640)
REAL diffusion-domain sample stats:
mean percentiles: [-2.44724512 -0.86997034 -0.33465129 -0.07803437 -0.00626281  0.06138756
  0.31676409  1.199641    1.81034207]
std percentiles: [0.25688761 0.28492849 0.32681829 0.49457722 0.66828385 0.89903714
 1.73293952 2.93165534 3.97933555]
maxabs percentiles: [ 1.14877844  1.60279977  2.08257633  2.9498052   3.83619595  5.70871043
  8.44314337 10.16624984 15.86707115]
dataset size: 1800
num batches: 56
epoch 1/60 loss=1.008686
epoch 2/60 loss=0.923693
epoch 3/60 loss=0.832881
epoch 4/60 loss=0.761015
epoch 5/60 loss=0.710921
epoch 6/60 loss=0.668548
epoch 7/60 loss=0.635749
epoch 8/60 loss=0.605932
epoch 9/60 loss=0.579105
epoch 10/60 loss=0.5